In [ ]:
import datetime
import io
import time
import pandas as pd
import requests
import yfinance as yf

# ---------------------------------------------------------
# 1. DATOS ESTRUCTURADOS: CO2 de la NOAA (Semanal)
# ---------------------------------------------------------
print("Descargando CO2 semanal de NOAA...")
noaa_url = "https://gml.noaa.gov/webdata/ccgg/trends/co2/co2_weekly_mlo.csv"
res = requests.get(noaa_url)

# El CSV de NOAA tiene comentarios iniciales marcados con '#'
noaa_lines = [
    line for line in res.text.splitlines() if not line.strip().startswith("#")
]
df_co2 = pd.read_csv(io.StringIO("\n".join(noaa_lines)))

# Formatear fecha y filtrar datos válidos (evitar valores centinela -999.99)
df_co2["date"] = pd.to_datetime(
    df_co2[["year", "month", "day"]].astype(str).agg("-".join, axis=1)
)
df_co2 = df_co2[df_co2["average"] > 0][["date", "average"]].rename(
    columns={"average": "co2_ppm"}
)

# ---------------------------------------------------------
# 2. DATOS ESTRUCTURADOS: Precio del Brent (Semanal)
# ---------------------------------------------------------
print("Descargando precios de petróleo Brent...")
brent = yf.download(
    "BZ=F", start="2013-01-01", end="2025-01-01", interval="1wk"
)["Close"]
df_brent = brent.reset_index()
df_brent.columns = ["date", "brent_price"]
df_brent["date"] = pd.to_datetime(df_brent["date"]).dt.tz_localize(None)

# ---------------------------------------------------------
# 3. ALINEACIÓN TEMPORAL DE SERIES NUMÉRICAS
# ---------------------------------------------------------
# Re-muestrear al final de cada semana (domingo)
df_merged = pd.merge_asof(
    df_brent.sort_values("date"),
    df_co2.sort_values("date"),
    on="date",
    direction="backward",
)

# ---------------------------------------------------------
# 4. DATOS NO ESTRUCTURADOS: Instantánea de Wikipedia
# ---------------------------------------------------------
WIKI_ENDPOINT = "https://en.wikipedia.org/w/api.php"
ARTICLE_TITLE = "Climate_change"  # O 'Cambio_climático' para español
HEADERS = {"User-Agent": "ResearchDataProject/1.0 (academic_nlp_study)"}
wiki_session = requests.Session()
wiki_session.headers.update(HEADERS)


def get_wiki_lead_at_date(title, target_date, max_retries=5):
    """Obtiene el texto de la introducción vigente en una fecha concreta.

    Reintenta con backoff cuando la API responde 429 (rate limit),
    respetando el header Retry-After en vez de descartar el resultado.
    """
    iso_timestamp = target_date.strftime("%Y-%m-%dT23:59:59Z")
    params = {
        "action": "query",
        "prop": "revisions",
        "titles": title,
        "rvlimit": 1,
        "rvstart": iso_timestamp,  # Última revisión anterior a este corte
        "rvdir": "older",
        "rvsection": 0,  # Sólo la sección 0 (la entradilla/resumen)
        "rvprop": "content|timestamp|ids",
        "format": "json",
    }
    for attempt in range(max_retries):
        try:
            resp = wiki_session.get(WIKI_ENDPOINT, params=params, timeout=15)
            if resp.status_code == 429:
                wait = int(resp.headers.get("Retry-After", 5))
                print(f"    [rate limit] esperando {wait}s antes de reintentar...")
                time.sleep(wait)
                continue
            resp.raise_for_status()
            data = resp.json()
            pages = data.get("query", {}).get("pages", {})
            page = next(iter(pages.values()))
            rev = page.get("revisions", [{}])[0]
            return {
                "rev_id": rev.get("revid"),
                "rev_timestamp": rev.get("timestamp"),
                "lead_text": rev.get("*", ""),
            }
        except requests.RequestException as e:
            print(f"    [error] intento {attempt + 1} falló para {target_date.date()}: {e}")
            time.sleep(2 * (attempt + 1))
    print(f"    [sin datos] {target_date.date()} tras {max_retries} intentos")
    return {"rev_id": None, "rev_timestamp": None, "lead_text": ""}


n_weeks = len(df_merged)
print(f"Extrayendo revisiones de Wikipedia para {n_weeks} semanas...")

wiki_records = []
n_ok = 0
n_fail = 0
start_time = time.time()
PROGRESS_EVERY = 10  # imprimir progreso cada N semanas

for idx, row in df_merged.iterrows():
    date_val = row["date"]
    rev_info = get_wiki_lead_at_date(ARTICLE_TITLE, date_val)
    wiki_records.append(rev_info)

    if rev_info["rev_id"] is not None:
        n_ok += 1
    else:
        n_fail += 1

    done = idx + 1
    if done % PROGRESS_EVERY == 0 or done == n_weeks:
        elapsed = time.time() - start_time
        rate = elapsed / done
        remaining = rate * (n_weeks - done)
        pct = 100 * done / n_weeks
        print(
            f"[{done}/{n_weeks} | {pct:5.1f}%] "
            f"semana {date_val.date()} | ok={n_ok} fail={n_fail} | "
            f"transcurrido={elapsed / 60:.1f}min restante_est={remaining / 60:.1f}min"
        )

    time.sleep(1)  # Respetar rate-limits de Wikipedia

print(f"Extracción de Wikipedia terminada: {n_ok} ok, {n_fail} fallidas.")

df_wiki = pd.DataFrame(wiki_records)
dataset_final = pd.concat(
    [df_merged.reset_index(drop=True), df_wiki], axis=1
)

# ---------------------------------------------------------
# 5. GUARDAR EL DATASET
# ---------------------------------------------------------
dataset_final.to_parquet("dataset_clima_multimodal.parquet", index=False)
print("¡Completado! Registros guardados:", len(dataset_final))

Descargando CO2 semanal de NOAA...


[*********************100%***********************]  1 of 1 completed

Descargando precios de petróleo Brent...
Extrayendo revisiones de Wikipedia para 627 semanas...


[10/627 |   1.6%] semana 2013-03-05 | ok=10 fail=0 | transcurrido=0.2min restante_est=12.4min
[20/627 |   3.2%] semana 2013-05-14 | ok=20 fail=0 | transcurrido=0.4min restante_est=12.7min
[30/627 |   4.8%] semana 2013-07-23 | ok=30 fail=0 | transcurrido=0.6min restante_est=12.7min
[40/627 |   6.4%] semana 2013-10-01 | ok=40 fail=0 | transcurrido=0.9min restante_est=12.8min
[50/627 |   8.0%] semana 2013-12-10 | ok=50 fail=0 | transcurrido=1.1min restante_est=12.8min
[60/627 |   9.6%] semana 2014-02-18 | ok=60 fail=0 | transcurrido=1.3min restante_est=12.5min
[70/627 |  11.2%] semana 2014-04-29 | ok=70 fail=0 | transcurrido=1.5min restante_est=12.3min
[80/627 |  12.8%] semana 2014-07-08 | ok=80 fail=0 | transcurrido=1.8min restante_est=12.1min


In [ ]:
import pandas as pd

df = pd.read_parquet("dataset_clima_multimodal.parquet")

print("Shape:", df.shape)
print("\nColumnas y tipos:")
print(df.dtypes)

print("\nNulos por columna:")
print(df.isna().sum())

print("\nPrimeras filas:")
display(df.head())

print("\nEjemplo de lead_text (fila 0):")
print(df.loc[0, "lead_text"][:500], "...")

df.describe(include="all").T

Shape: (627, 6)

Columnas y tipos:
date             datetime64[ns]
brent_price             float64
co2_ppm                 float64
rev_id                  float64
rev_timestamp            object
lead_text                object
dtype: object

Nulos por columna:
date               0
brent_price        0
co2_ppm            0
rev_id           587
rev_timestamp    587
lead_text          0
dtype: int64

Primeras filas:


,date,brent_price,co2_ppm,rev_id,rev_timestamp,lead_text
0,2013-01-01,111.400002,394.72,530602112.0,2012-12-31T12:27:35Z,{{About|the current change in Earth's climate|...
1,2013-01-08,111.879997,395.60,531873864.0,2013-01-08T00:59:41Z,{{About|the current change in Earth's climate|...
2,2013-01-15,110.610001,396.21,531873864.0,2013-01-08T00:59:41Z,{{About|the current change in Earth's climate|...
3,2013-01-22,113.480003,396.01,533894306.0,2013-01-19T20:21:34Z,{{About|the current change in Earth's climate|...
4,2013-01-29,115.599998,395.99,535083671.0,2013-01-27T01:10:30Z,{{About|the current change in Earth's climate|...



Ejemplo de lead_text (fila 0):
{{About|the current change in Earth's climate|general discussion of how the climate can change | Climate change|other uses}}
{{bots|deny=Citation bot}}
{{pp-semi-protected|small=yes}}
{{Featured article}}
{{Multiple image|align=right|direction=vertical|width=220|image1=Global Temperature Anomaly 1880-2010 (Fig.A).gif|alt1=refer to caption|caption1=Global mean land-ocean temperature change from 1880–2011, relative to the 1951–1980 mean. The black line is the annual mean and the red line is the 5- ...


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
date,627,NaN,NaN,NaN,2019-01-01 00:00:00,2013-01-01 00:00:00,2016-01-01 12:00:00,2019-01-01 00:00:00,2021-12-31 12:00:00,2024-12-31 00:00:00,NaN
brent_price,627.0,NaN,NaN,NaN,72.689282,19.99,55.23,71.519997,86.195,123.209999,22.512312
co2_ppm,627.0,NaN,NaN,NaN,410.251738,393.52,402.605,410.2,417.66,427.94,8.866993
rev_id,40.0,NaN,NaN,NaN,788589818.675,530602112.0,620959306.0,758739873.0,927411983.75,1106229946.0,217977580.434965
rev_timestamp,40,37,2013-01-08T00:59:41Z,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
lead_text,627,24,,587,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df.lead_text.str.len().value_counts().sort_values(ascending=False)

lead_text
0        587
15851      5
15827      4
11603      4
20695      4
22001      3
20682      2
11339      2
19299      2
15838      1
21043      1
20539      1
20699      1
18932      1
20523      1
18983      1
19290      1
22029      1
21999      1
11384      1
11337      1
11737      1
11813      1
Name: count, dtype: int64